In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [1]:
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score
from sklearn.preprocessing import PowerTransformer

In [2]:
df = pd.read_csv('/content/drive/MyDrive/colab/mlpr/dataset.csv')

In [34]:
display(df.columns)

Index(['timestamp', 'pm25_ugm3', 'pm10_ugm3', 'no_ugm3', 'no2_ugm3', 'nox_ppb',
       'nh3_ugm3', 'so2_ugm3', 'co_mgm3', 'ozone_ugm3', 'at_c', 'rh_pct',
       'ws_ms', 'wd_deg', 'rf_mm', 'tot_rf_mm', 'sr_wm2', 'state', 'city',
       'station', 'station_id', 'latitude', 'longitude', 'era5_sw_down_wm2',
       'era5_dewpoint_k', 'era5_pressure_pa', 'era5_temp_k',
       'era5_cloud_cover', 'era5_precip_m', 'era5_u10_ms', 'era5_v10_ms',
       'era5_temp_c', 'era5_dewpoint_c', 'era5_pressure_hpa', 'era5_precip_mm',
       'era5_wind_speed_ms', 'era5_wind_dir_deg', 'era5_vpd_kpa',
       'era5_rh_pct', 'solar_altitude_deg', 'solar_zenith_deg', 'cos_zenith'],
      dtype='object')

In [9]:
display(df.head())

,pm25_ugm3,pm10_ugm3,no_ugm3,no2_ugm3,nox_ppb,nh3_ugm3,so2_ugm3,co_mgm3,ozone_ugm3,latitude,...,solar_zenith_deg,cos_zenith,station_id_encoded,station_encoded,state_encoded,city_encoded,year,month,day,hour
0,19.084999,38.794998,14.382500,4.947500,14.322500,10.155000,3.480000,0.762500,4.765000,13.204880,...,85.332840,0.081367,360.158081,360.158081,269.334015,360.158081,2023,1,1,7
1,45.410000,64.642502,5.647500,13.365000,18.982500,5.905000,1.222500,0.292500,11.787500,11.875000,...,87.967094,0.035473,259.040100,259.040100,221.699615,259.040100,2023,1,1,7
2,77.750000,164.000000,12.100000,67.025002,45.474998,53.200001,44.349998,0.172500,4.533333,26.088131,...,83.800438,0.107992,140.013062,140.013062,112.064301,140.013062,2023,1,1,7
3,52.500000,79.000000,4.012735,23.902308,12.866176,9.081314,4.175000,0.730566,3.733333,27.308329,...,86.959023,0.053050,26.129847,26.129847,112.064301,26.129847,2023,1,1,7
4,62.250000,186.250000,0.425000,30.600000,16.600000,25.197840,12.875000,0.375000,36.866665,25.376776,...,84.595512,0.094186,104.376892,104.376892,112.064301,104.376892,2023,1,1,7


In [3]:
#df.drop(columns=['latitude', 'longitude', 'station_id', 'station', 'state', 'city'],  inplace=True)
from sklearn.preprocessing import LabelEncoder


target_encoders = {}

for col in ['station_id', 'station', 'state', 'city']:
    if col in df.columns:
        target_encoders[col] = df.groupby(col)['sr_wm2'].mean().to_dict()

# Step 2: Apply encoding to entire dataset
global_mean = df['sr_wm2'].mean()
for col in ['station_id', 'station', 'state', 'city']:
    if col in df.columns:
        df[f'{col}_encoded'] = df[col].map(target_encoders[col])
        df[f'{col}_encoded'] = df[f'{col}_encoded'].fillna(global_mean)


float_cols = df.select_dtypes(include=['float64']).columns
df[float_cols] = df[float_cols].astype('float32')

df['timestamp'] = pd.to_datetime(df['timestamp'])

print(df.info())

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 2549291 entries, 0 to 2549290
Data columns (total 46 columns):
 #   Column              Dtype         
---  ------              -----         
 0   timestamp           datetime64[ns]
 1   pm25_ugm3           float32       
 2   pm10_ugm3           float32       
 3   no_ugm3             float32       
 4   no2_ugm3            float32       
 5   nox_ppb             float32       
 6   nh3_ugm3            float32       
 7   so2_ugm3            float32       
 8   co_mgm3             float32       
 9   ozone_ugm3          float32       
 10  at_c                float32       
 11  rh_pct              float32       
 12  ws_ms               float32       
 13  wd_deg              float32       
 14  rf_mm               float32       
 15  tot_rf_mm           float32       
 16  sr_wm2              float32       
 17  state               object        
 18  city                object        
 19  station             object        
 20  st

In [4]:
# Make sure rows are in chronological order
df = df.sort_values("timestamp").reset_index(drop=True)

# Feature engineering
df["year"] = df["timestamp"].dt.year
df["month"] = df["timestamp"].dt.month
df["day"] = df["timestamp"].dt.day
df["hour"] = df["timestamp"].dt.hour

# Drop obvious redundancies
cols_to_drop = [
    'timestamp',                    # Already extracted as year/month/day/hour
    'at_c', 'rh_pct', 'ws_ms', 'wd_deg', 'rf_mm', 'tot_rf_mm',  # Duplicated in ERA5
    'station_id', 'station', 'state', 'city',  # Keep encoded versions only
    'era5_temp_k', 'era5_dewpoint_k', 'era5_pressure_pa',  # Keep _c and _hpa versions
    'era5_u10_ms', 'era5_v10_ms',   # Use era5_wind_speed_ms instead
    'era5_precip_m',                # Keep era5_precip_mm
    'solar_altitude_deg',           # Keep solar_zenith_deg (more informative for prediction)
    'era5_sw_down_wm2',             # Target leakage!
]

df.drop(columns=cols_to_drop, inplace=True)

In [5]:
# 1. Cyclical Time Encoding (Fixing the 11 PM to Midnight jump)
df['hour_sin'] = np.sin(df['hour'] * (2. * np.pi / 24)).astype('float32')
df['hour_cos'] = np.cos(df['hour'] * (2. * np.pi / 24)).astype('float32')
df['month_sin'] = np.sin((df['month'] - 1) * (2. * np.pi / 12)).astype('float32')
df['month_cos'] = np.cos((df['month'] - 1) * (2. * np.pi / 12)).astype('float32')

# Drop the raw hour and month columns as they are no longer needed
df = df.drop(columns=['hour', 'month'])

# 2. Sort Chronologically AND Geographically (CRITICAL)
df = df.sort_values(by=['station_id_encoded', 'year', 'hour_sin'])

# 3. Create Lag and Rolling Features per station
# Example: What was the cloud cover 24 hours ago?
# df['cloud_cover_lag24'] = df.groupby('station_id_encoded')['era5_cloud_cover'].shift(24).astype('float32')

# # Example: 6-hour moving average of PM2.5 (Aerosol scattering)
# df['pm25_rolling_6h'] = df.groupby('station_id_encoded')['pm25_ugm3'].transform(
#     lambda x: x.rolling(window=6, min_periods=1).mean()
# ).astype('float32')

# # What was the solar radiation exactly 24 hours ago?
# df['sr_wm2_lag24'] = df.groupby('station_id_encoded')['sr_wm2'].shift(24).astype('float32')

# # What was the temperature 24 hours ago?
# df['era5_temp_c_lag24'] = df.groupby('station_id_encoded')['era5_temp_c'].shift(24).astype('float32')

# Drop the rows at the top of each station's timeline that now have NaNs due to shifting
df = df.dropna()

In [6]:

# 1. Define the columns that should NEVER be lagged
columns_to_ignore = [
    'timestamp', 'state', 'city', 'station', 'station_id', 
    'station_id_encoded', 'latitude', 'longitude', 
    'solar_altitude_deg', 'solar_zenith_deg', 'cos_zenith', 
    'hour_sin', 'hour_cos', 'month_sin', 'month_cos', 'year'
]

# 2. Dynamically create a list of all columns that are safe to lag
# This will capture all your pm25, era5 weather vars, and the target variable (sr_wm2)
columns_to_lag = [col for col in df.columns if col not in columns_to_ignore]

print(f"Adding lags for {len(columns_to_lag)} dynamic features...")

# 3. Ensure data is sorted geographically and chronologically first to prevent data bleeding
df = df.sort_values(by=['station_id_encoded', 'year', 'hour_sin'])

# 4. Loop through the safe columns and generate 1-hour and 24-hour lags
for col in columns_to_lag:
    # 1-hour lag
    df[f'{col}_lag1'] = df.groupby('station_id_encoded')[col].shift(1).astype('float32')
    
    # 24-hour lag
    df[f'{col}_lag24'] = df.groupby('station_id_encoded')[col].shift(24).astype('float32')

# 5. Drop the rows at the very top of each station's timeline that now have NaNs
print(f"Original row count: {len(df)}")
df = df.dropna()
print(f"Row count after dropping initial NaNs: {len(df)}")

Adding lags for 23 dynamic features...
Original row count: 2549291
Row count after dropping initial NaNs: 2543891


In [7]:
# 1. Normalize the Target Variable (Yeo-Johnson Power Transform)
pt = PowerTransformer(method='yeo-johnson')
df['sr_wm2_scaled'] = pt.fit_transform(df[['sr_wm2']]).astype('float32')

# 2. Chronological Split (e.g., Train on years < 2023, Test on 2023)
train_df = df[df['year'] < 2025]
test_df = df[df['year'] >= 2025]

# Separate features (X) and target (y)
features = [col for col in df.columns if col not in ['sr_wm2', 'sr_wm2_scaled', 'year']]

X_train = train_df[features]
y_train = train_df['sr_wm2_scaled']

X_test = test_df[features]
y_test = test_df['sr_wm2_scaled']

print(f"Training rows: {len(X_train)} | Testing rows: {len(X_test)}")

/usr/local/lib/python3.12/dist-packages/numpy/_core/_methods.py:188: RuntimeWarning: overflow encountered in multiply
  x = um.multiply(x, x, out=x)


Training rows: 1694822 | Testing rows: 849069


In [8]:
import lightgbm as lgb

# 1. Set up LightGBM Dataset
train_data = lgb.Dataset(X_train, label=y_train)
test_data = lgb.Dataset(X_test, label=y_test, reference=train_data)

# 2. Define Model Parameters
params = {
    'objective': 'regression',
    'metric': 'rmse',
    'boosting_type': 'gbdt',
    'learning_rate': 0.05,
    'num_leaves': 63,
    'max_depth': 8,
    'feature_fraction': 0.8, # Use 80% of features per tree to prevent overfitting
    'n_jobs': -1 # Use all available CPU cores
}

# 3. Train the Model
print("Starting training...")
model = lgb.train(
    params,
    train_data,
    num_boost_round=1000,
    valid_sets=[train_data, test_data],
    callbacks=[lgb.early_stopping(stopping_rounds=50)]
)

# 4. Predict and Inverse Transform
preds_scaled = model.predict(X_test, num_iteration=model.best_iteration)

# We must transform predictions back to original W/m2 scale for accurate metrics
preds_original_scale = pt.inverse_transform(preds_scaled.reshape(-1, 1))
y_test_original_scale = pt.inverse_transform(y_test.values.reshape(-1, 1))

# 5. Evaluate
mae = mean_absolute_error(y_test_original_scale, preds_original_scale)
rmse = np.sqrt(mean_squared_error(y_test_original_scale, preds_original_scale))
r2 = r2_score(y_test_original_scale, preds_original_scale)

print(f"\n--- Model Performance ---")
print(f"MAE:  {mae:.2f} W/m²")
print(f"RMSE: {rmse:.2f} W/m²")
print(f"R²:   {r2:.4f}")

Starting training...
[LightGBM] [Info] Auto-choosing col-wise multi-threading, the overhead of testing was 1.473433 seconds.
You can set `force_col_wise=true` to remove the overhead.
[LightGBM] [Info] Total Bins 17033
[LightGBM] [Info] Number of data points in the train set: 1694822, number of used features: 77
[LightGBM] [Info] Start training from score 0.034153
Training until validation scores don't improve for 50 rounds
Early stopping, best iteration is:
[410]	training's rmse: 0.330071	valid_1's rmse: 0.359243

--- Model Performance ---
MAE:  44.35 W/m²
RMSE: 90.62 W/m²
R²:   0.8194


/usr/local/lib/python3.12/dist-packages/sklearn/utils/validation.py:2739: UserWarning: X does not have valid feature names, but PowerTransformer was fitted with feature names
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/sklearn/utils/validation.py:2739: UserWarning: X does not have valid feature names, but PowerTransformer was fitted with feature names
  warnings.warn(
